### Ingest the 4 KPI CSVs into Bronze Delta tables

In [0]:
%sql
USE CATALOG policyiq;

CREATE OR REPLACE TABLE bronze.branch_dim AS
SELECT * FROM read_files(
  '/Volumes/policyiq/sources/kpi_raw_files/branch_dim.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE bronze.policy_registry AS
SELECT * FROM read_files(
  '/Volumes/policyiq/sources/kpi_raw_files/policy_registry_seed.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE bronze.kpi_registry AS
SELECT * FROM read_files('/Volumes/policyiq/sources/kpi_raw_files/kpi_registry_seed.csv', format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE bronze.kpi_actuals AS
SELECT * FROM read_files('/Volumes/policyiq/sources/kpi_raw_files/kpi_actuals.csv', format => 'csv', header => true, inferSchema => true);



In [0]:
%sql
SELECT 'branch_dim' t, count(*) FROM  policyiq.bronze.branch_dim
UNION ALL SELECT 'policy_registry', count(*) FROM  policyiq.bronze.policy_registry
UNION ALL SELECT 'kpi_registry', count(*) FROM  policyiq.bronze.kpi_registry
UNION ALL SELECT 'kpi_actuals', count(*) FROM  policyiq.bronze.kpi_actuals;

### Land the raw PDF bytes into a Bronze table

In [0]:
%sql
CREATE OR REPLACE TABLE policyiq.bronze.policy_documents_raw AS
SELECT
  path,
  regexp_extract(path, '([^/]+)$', 1)      AS file_name,
  content,
  length,
  modificationTime
FROM read_files(
  '/Volumes/policyiq/sources/policy_pdf/',
  format => 'binaryFile'
);

SELECT file_name, length FROM  policyiq.bronze.policy_documents_raw ORDER BY file_name;

### Map each file to its policy_id

In [0]:
%sql
CREATE OR REPLACE TABLE  policyiq.bronze.policy_documents_mapped AS
SELECT
  *,
  CASE
    WHEN lower(file_name) LIKE '%ekyc%'                          THEN 'EKYC_2026'
    WHEN lower(file_name) LIKE '%aml%'                           THEN 'AML_MLTF_2026'
    WHEN lower(file_name) LIKE '%credit%'                        THEN 'CRM_2016'
    WHEN lower(file_name) LIKE '%cyber%'                         THEN 'CYBERSEC_2026'
    WHEN lower(file_name) LIKE '%hr%' OR lower(file_name) LIKE '%leave%' THEN 'HR_LEAVE_2026'
    ELSE 'UNKNOWN'
  END AS policy_id
FROM  policyiq.bronze.policy_documents_raw;

SELECT file_name, policy_id FROM  policyiq.bronze.policy_documents_mapped ORDER BY policy_id;

### Parse all 5 PDFs

In [0]:
%sql
CREATE OR REPLACE TABLE policyiq.bronze.policy_documents_parsed AS
SELECT
  file_name,
  policy_id,
  ai_parse_document(content, map('version', '2.0')) AS parsed
FROM  policyiq.bronze.policy_documents_mapped;

In [0]:
%sql
select * from policyiq.bronze.policy_documents_parsed

In [0]:
%sql
SELECT
  file_name,
  policy_id,
  size(from_json(to_json(parsed:document.pages), 'array<variant>')) AS page_count
FROM  policyiq.bronze.policy_documents_parsed
ORDER BY policy_id;

In [0]:
%sql
SELECT 'branch_dim' AS dataset, count(*) AS row_count FROM  policyiq.bronze.branch_dim
UNION ALL SELECT 'policy_registry', count(*) FROM  policyiq.bronze.policy_registry
UNION ALL SELECT 'kpi_registry', count(*) FROM  policyiq.bronze.kpi_registry
UNION ALL SELECT 'kpi_actuals', count(*) FROM  policyiq.bronze.kpi_actuals
UNION ALL SELECT 'policy_documents_parsed', count(*) FROM  policyiq.bronze.policy_documents_parsed;


In [0]:
%sql
select * from policyiq.bronze.branch_dim

In [0]:
%sql
select * from policyiq.bronze.kpi_actuals

In [0]:
%sql
select * from policyiq.bronze.kpi_registry

In [0]:
%sql
select * from policyiq.bronze.policy_registry

### Add real column and table comments

In [0]:
%sql
COMMENT ON TABLE policyiq.bronze.branch_dim IS
'Branch master data - one row per physical bank branch, with location and classification attributes.';

ALTER TABLE policyiq.bronze.branch_dim ALTER COLUMN district COMMENT 'District (finer-grained than division) where the branch is located.';
ALTER TABLE policyiq.bronze.branch_dim ALTER COLUMN branch_type COMMENT 'Urban, Semi-Urban, or Rural classification of the branch location.';
ALTER TABLE policyiq.bronze.branch_dim ALTER COLUMN branch_category COMMENT 'Business category of the branch: Corporate, General, SME, or Agent Banking Hub.';
ALTER TABLE policyiq.bronze.branch_dim ALTER COLUMN opening_date COMMENT 'Date the branch was opened.';
ALTER TABLE policyiq.bronze.branch_dim ALTER COLUMN employee_count COMMENT 'Number of employees currently working at the branch.';

COMMENT ON TABLE policyiq.bronze.policy_registry IS
'Master list of the 5 policy documents governing this bank - what each one is, who issued it, and whether it is a real regulatory document or a synthetic/demonstration one.';

ALTER TABLE policyiq.bronze.policy_registry ALTER COLUMN policy_name COMMENT 'Full official name of the policy document.';
ALTER TABLE policyiq.bronze.policy_registry ALTER COLUMN issuing_authority COMMENT 'The organization that issued/published this policy, e.g. Bangladesh Bank.';
ALTER TABLE policyiq.bronze.policy_registry ALTER COLUMN version COMMENT 'Version or edition of the policy document.';
ALTER TABLE policyiq.bronze.policy_registry ALTER COLUMN effective_date COMMENT 'Date the policy became effective.';
ALTER TABLE policyiq.bronze.policy_registry ALTER COLUMN source_file_path COMMENT 'Path to the original source PDF file.';
ALTER TABLE policyiq.bronze.policy_registry ALTER COLUMN is_synthetic COMMENT 'TRUE if this is a demonstration/fictitious document (currently only the HR Leave Policy), FALSE if it is a real regulatory document.';

COMMENT ON TABLE policyiq.bronze.kpi_registry IS
'Definition of every KPI tracked in this system - what it measures, which policy and exact clause governs it, and the threshold logic.';

ALTER TABLE policyiq.bronze.kpi_registry ALTER COLUMN notes COMMENT 'Explanation of how this threshold was derived from the source policy text, including any cases where the numeric threshold was inferred rather than stated explicitly in the policy.';